In [1]:
import os
from langchain.agents import create_agent
from dotenv import load_dotenv

c:\Delta Training Local\gen-ai\Langchain agents\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise ValueError("GROQ_API_KEY is not set. Add it to your environment or .env file.")
os.environ["GROQ_API_KEY"] = api_key
os.environ["LANGFUSE_SECRET_KEY"] = os.getenv("LANGFUSE_SECRET_KEY")
os.environ["LANGFUSE_PUBLIC_KEY"] = os.getenv("LANGFUSE_PUBLIC_KEY")
os.environ["LANGFUSE_BASE_URL"] = os.getenv("LANGFUSE_BASE_URL")

In [3]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
)

#PDF Parsing

In [ ]:
from langchain_pymupdf4llm import PyMuPDF4LLMLoader


In [ ]:
os.environ["HUGGINGFACEHUB_API_TOKEN"]= os.getenv("HUGGINGFACEHUB_API_TOKEN")

In [9]:
from langchain_huggingface.embeddings import HuggingFaceEndpointEmbeddings

In [10]:
embeddings = HuggingFaceEndpointEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2"
)

In [13]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db",
)

In [14]:
from uuid import uuid4

In [16]:
ret = vector_store.as_retriever(
    search_type="mmr", search_kwargs={"k": 2, "fetch_k": 5}
)
# res=retriever.invoke("what is ML")

In [18]:
prompt = """Use the following retrieved documents to answer the question: {question}
{retrieved_docs}
Answer:"""


In [19]:
from langchain.tools import tool


In [20]:
@tool("pdf_parser")
def pdf_parser(path: str):
    """Parse a PDF file and return its content as a string."""
    loader = PyMuPDF4LLMLoader(path)
    docs = loader.load()
    vector_store.add_documents(documents=docs, ids=[str(uuid4()) for _ in range(len(docs))])
    return f"Parsed {len(docs)} documents from {path} and added to vector store."

In [21]:
@tool("retriever")
def retriever(query: str):
    """Retrieve relevant documents based on the query."""
    res = ret.invoke(query)
    response =llm.invoke(prompt.format(question=query, retrieved_docs=res))
    return response


In [ ]:
# @tool("summarizer")
# def summarizer(text: str):
#     """Summarize the given text."""
#     summary_prompt = f"Summarize the following text:\n\n{text}\n\nSummary: in to a few sentences."
#     summary = llm.invoke(summary_prompt)
#     return summary
    

In [23]:
from langfuse.langchain import CallbackHandler
from langfuse import get_client
langfuse = get_client()
langfuse_handler = CallbackHandler()

In [29]:
agent=create_agent(
    model=llm,
    tools=[pdf_parser, retriever],
    system_prompt="You are an assistant that can parse PDF documents and retrieve information from them to answer questions.",
)

In [30]:
result = agent.invoke({
    "messages": [{"role": "user", "content": "Please parse the document at 'samplee.pdf' and then answer the question: What is ML? path=samplee.pdf"}]
},config={"callbacks": [langfuse_handler]})
result#["messages"][-1].content

{'messages': [HumanMessage(content="Please parse the document at 'samplee.pdf' and then answer the question: What is ML? path=samplee.pdf", additional_kwargs={}, response_metadata={}, id='39143794-43b3-4f99-9b23-83372bc6d256'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants me to parse the document at \'samplee.pdf\' and then answer the question "What is ML?" So first, I need to use the pdf_parser function to get the content of the PDF. The path parameter is given as \'samplee.pdf\', so I\'ll call that function with that path. Once I have the content, I should look through it to find information about ML. If the PDF explains what ML is, I can summarize that. If not, maybe I need to use the retriever function to find additional documents. But since the user specifically asked to parse the given PDF, I should focus on that first. Let me check the functions available again. The pdf_parser will return the content as a string, so I\'ll process that to 

In [31]:
result["messages"][-1].content

'Machine Learning (ML) is a field of study where algorithms learn patterns from data to make predictions or decisions without being explicitly programmed for specific tasks. Here\'s a structured breakdown based on the provided documents:\n\n1. **Core Components of ML**:\n   - **Training Data**: Historical examples or past experiences used to train models.\n   - **Target Concept**: A hypothesis (function) that maps inputs to outputs, learned from the training data.\n   - **Data Objects**: Instances used to test and validate the trained model\'s performance.\n\n2. **Supervised Learning Focus**:\n   - ML models in supervised learning aim to find the best **hypothesis** (a function approximating the target concept) from a **hypothesis space** (all possible legal hypotheses). This hypothesis maps inputs to outputs using labeled datasets (input-output pairs).\n\n3. **Hypothesis Representation**:\n   - Hypotheses are defined by constraints on input attributes (e.g., Sky, AirTemp).\n   - A hyp